In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings


load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)



In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('speech.txt')
docs = loader.load()
docs


C:\Users\umara\AppData\Local\Temp\ipykernel_10432\1883200319.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'speech.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications powered by Large Language Models (LLMs). It provides tools and components for connecting LLMs with external data sources, APIs, databases, prompts, memory, and other services. Developers can use LangChain to build applications such as AI chatbots, Retrieval-Augmented Generation (RAG) systems, question-answering applications, and AI agents that can use tools to perform different tasks. It supports popular LLM providers such as OpenAI, Anthropic, and Google, making it easier to develop flexible and powerful generative AI applications.')]

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

final_documents = text_splitter.split_documents(docs)

final_documents

[Document(metadata={'source': 'speech.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications powered by Large Language Models (LLMs). It provides tools and components for connecting LLMs with external data sources, APIs, databases, prompts, memory, and other services. Developers can use LangChain to build applications such as AI chatbots, Retrieval-Augmented Generation (RAG) systems, question-answering applications, and AI agents that can use tools to perform different tasks. It supports popular'),
 Document(metadata={'source': 'speech.txt'}, page_content='to perform different tasks. It supports popular LLM providers such as OpenAI, Anthropic, and Google, making it easier to develop flexible and powerful generative AI applications.')]

In [5]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(final_documents, embeddings)

db

In [6]:
query = "Developers can use LangChain to build applications"

# Similarity search
retrieve_result = db.similarity_search(query)

print(retrieve_result)

[Document(id='f3ab28b1-0f9c-4932-93e1-61cd4b899e02', metadata={'source': 'speech.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications powered by Large Language Models (LLMs). It provides tools and components for connecting LLMs with external data sources, APIs, databases, prompts, memory, and other services. Developers can use LangChain to build applications such as AI chatbots, Retrieval-Augmented Generation (RAG) systems, question-answering applications, and AI agents that can use tools to perform different tasks. It supports popular'), Document(id='f26a153c-467c-48cd-8766-d059b22da698', metadata={'source': 'speech.txt'}, page_content='to perform different tasks. It supports popular LLM providers such as OpenAI, Anthropic, and Google, making it easier to develop flexible and powerful generative AI applications.')]


In [7]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='f3ab28b1-0f9c-4932-93e1-61cd4b899e02', metadata={'source': 'speech.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications powered by Large Language Models (LLMs). It provides tools and components for connecting LLMs with external data sources, APIs, databases, prompts, memory, and other services. Developers can use LangChain to build applications such as AI chatbots, Retrieval-Augmented Generation (RAG) systems, question-answering applications, and AI agents that can use tools to perform different tasks. It supports popular'),
  np.float32(0.39153725)),
 (Document(id='f26a153c-467c-48cd-8766-d059b22da698', metadata={'source': 'speech.txt'}, page_content='to perform different tasks. It supports popular LLM providers such as OpenAI, Anthropic, and Google, making it easier to develop flexible and powerful generative AI applications.'),
  np.float32(0.48327458))]

In [10]:
embedding_vector_query = embeddings.embed_query(query)


In [16]:
docs_and_scores = db.similarity_search_by_vector(embedding_vector_query)
docs_and_scores[0]

Document(id='f3ab28b1-0f9c-4932-93e1-61cd4b899e02', metadata={'source': 'speech.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications powered by Large Language Models (LLMs). It provides tools and components for connecting LLMs with external data sources, APIs, databases, prompts, memory, and other services. Developers can use LangChain to build applications such as AI chatbots, Retrieval-Augmented Generation (RAG) systems, question-answering applications, and AI agents that can use tools to perform different tasks. It supports popular')

In [17]:
### Saving And Loading

# Save FAISS database locally
db.save_local("faiss_index")

In [18]:
new_db = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [19]:
docs = new_db.similarity_search(query)

docs

[Document(id='f3ab28b1-0f9c-4932-93e1-61cd4b899e02', metadata={'source': 'speech.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications powered by Large Language Models (LLMs). It provides tools and components for connecting LLMs with external data sources, APIs, databases, prompts, memory, and other services. Developers can use LangChain to build applications such as AI chatbots, Retrieval-Augmented Generation (RAG) systems, question-answering applications, and AI agents that can use tools to perform different tasks. It supports popular'),
 Document(id='f26a153c-467c-48cd-8766-d059b22da698', metadata={'source': 'speech.txt'}, page_content='to perform different tasks. It supports popular LLM providers such as OpenAI, Anthropic, and Google, making it easier to develop flexible and powerful generative AI applications.')]